# Environnement creation

In [1]:
%%bash
# Create the virtual environments
python -m venv env_qiskit_241
python -m venv env_qiskit_250

# Upgrade pip in both environments
./env_qiskit_241/bin/python -m pip install --upgrade pip -q
./env_qiskit_250/bin/python -m pip install --upgrade pip -q

# Install specific Qiskit versions
./env_qiskit_241/bin/python -m pip install qiskit==2.4.1 -q
./env_qiskit_250/bin/python -m pip install qiskit==2.5.0 -q

echo "Environments configured."

Couldn't find program: 'bash'


## Sabre comparaison

In [1]:
import os
import sys
import time
import glob
import pandas as pd
from qiskit import qasm2
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler import StagedPassManager
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke, FakeNighthawk

def execute_transpile_benchmark(circuits_dir: str, output_csv: str = "transpile_benchmark.csv", number_seeds: int = 5, base_seed: int = 42):
    if not os.path.isdir(circuits_dir):
        sys.exit(f"FATAL: Target directory '{circuits_dir}' does not exist.")
        
    qasm_files = glob.glob(os.path.join(circuits_dir, "*.qasm"))
    if not qasm_files:
        sys.exit(f"FATAL: No .qasm files found in '{circuits_dir}'.")
        
    circuits = {}
    for f in qasm_files:
        file_name = os.path.basename(f)
        try:
            circuits[file_name] = qasm2.load(
                f,
                custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS,
                custom_classical=qasm2.LEGACY_CUSTOM_CLASSICAL,
            )
        except Exception as e:
            print(f"[!] WARNING: Bypassing '{file_name}'. QASM parsing failed: {e}")
            continue

    if not circuits:
        sys.exit(f"FATAL: No valid QASM files could be parsed in '{circuits_dir}'. Pipeline terminated.")
        
    print(f"[+] Successfully loaded {len(circuits)} valid circuit(s) for benchmarking.")

    hardware_targets = {
        "Sherbrooke_127Q": FakeSherbrooke(),
        "Nighthawk_120Q": FakeNighthawk()
    }

    seeds = [base_seed + i for i in range(number_seeds)]
    records = []

    for backend_name, backend in hardware_targets.items():
        print(f"\n--- Initializing Target Architecture: {backend_name} ---")
        hw_qubits = backend.num_qubits 

        for circ_name, qc in circuits.items():
            logical_qubits = qc.num_qubits
            print(f"\nTarget Circuit: {circ_name} | Logical Qubits: {logical_qubits} | Hardware Limits: {hw_qubits}")
            
            if logical_qubits > hw_qubits:
                print(f"    [!] Violation: {circ_name} exceeds {backend_name} hardware limits. Bypassing.")
                continue

            for seed in seeds:
                try:
                    start_time = time.time()
                    
                    # Generate optimal preset configuration
                    preset_pm = generate_preset_pass_manager(
                        optimization_level=3,
                        backend=backend,
                        seed_transpiler=seed
                    )

                    # 1. Execute and isolate the INIT stage
                    init_pm = StagedPassManager(stages=["init"], init=preset_pm.init)
                    qc_init = init_pm.run(qc)

                    # 2. Execute LAYOUT and ROUTING stages on the init output
                    route_pm = StagedPassManager(
                        stages=["layout", "routing"],
                        layout=preset_pm.layout,
                        routing=preset_pm.routing
                    )
                    transpiled_qc = route_pm.run(qc_init)
                    
                    exec_time = time.time() - start_time
                    
                    ops = transpiled_qc.count_ops()
                    total_physical_gates = sum(ops.values())
                    swap_count = ops.get("swap", 0)
                    total_depth = transpiled_qc.depth()
                    depth_2q = transpiled_qc.depth(filter_function=lambda x: x.operation.num_qubits >= 2)

                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": round(exec_time, 4),
                        "Total Gates": total_physical_gates,
                        "SWAP Count": swap_count,
                        "Total Depth": total_depth,
                        "2Q Depth": depth_2q,
                        "Gate Breakdown": str(dict(ops)),
                        "Error": None
                    })
                    print(f"    [+] Seed {seed} compiled successfully in {exec_time:.2f}s. (SWAPs: {swap_count}, Depth: {total_depth})")
                    
                except Exception as e:
                    print(f"    [!] Failure during transpile on seed {seed}: {str(e)}")
                    records.append({
                        "Seed": seed,
                        "Backend": backend_name,
                        "Circuit": circ_name,
                        "Execution Time (s)": None,
                        "Total Gates": None,
                        "SWAP Count": None,
                        "Total Depth": None,
                        "2Q Depth": None,
                        "Gate Breakdown": None,
                        "Error": str(e)
                    })

    if not records:
        sys.exit("FATAL: No successful transpilation records generated. Exiting.")

    df_raw = pd.DataFrame(records)
    
    
    df_success = df_raw[df_raw["Error"].isnull()].copy()
    
    numeric_cols = ["Execution Time (s)", "Total Gates", "SWAP Count", "Total Depth", "2Q Depth"]
    for col in numeric_cols:
        df_success.loc[:, col] = pd.to_numeric(df_success[col])

    group_keys = ["Backend", "Circuit"]
    
    df_avg = df_success.groupby(group_keys)[numeric_cols].mean().round(2).reset_index()
    
    success_counts = df_success.groupby(group_keys).size().reset_index(name="Successful_Runs")
    df_avg = pd.merge(df_avg, success_counts, on=group_keys, how="left")
    df_avg["Total_Seeds"] = number_seeds

    final_columns = group_keys + ["Successful_Runs", "Total_Seeds"] + numeric_cols
    df_avg = df_avg[final_columns]

    avg_output = "averaged_" + output_csv
    df_avg.to_csv(avg_output, index=False)
    
    print(f"\nPipeline exhausted. Generated artifacts:")
    print(f"  2. {avg_output} (Statistical matrix)")
    
    return df_avg

if __name__ == "__main__":
    execute_transpile_benchmark(circuits_dir="circuits", output_csv="routing_only_benchmark_2_5_0.csv", number_seeds=5, base_seed=42)

[+] Successfully loaded 88 valid circuit(s) for benchmarking.

--- Initializing Target Architecture: Sherbrooke_127Q ---


c:\Users\axelp\.virtualenvs\qiskit_2_5_0\Lib\site-packages\qiskit_ibm_runtime\fake_provider\backends\nighthawk\fake_nighthawk.py:78: UserWarning: Properties of fake_nighthawk are not intended to represent typical nighthawk error values.
  warnings.warn(



Target Circuit: adder_n10.qasm | Logical Qubits: 10 | Hardware Limits: 127
    [+] Seed 42 compiled successfully in 0.06s. (SWAPs: 15, Depth: 223)
    [+] Seed 43 compiled successfully in 0.01s. (SWAPs: 15, Depth: 223)
    [+] Seed 44 compiled successfully in 0.01s. (SWAPs: 15, Depth: 223)
    [+] Seed 45 compiled successfully in 0.01s. (SWAPs: 15, Depth: 223)
    [+] Seed 46 compiled successfully in 0.01s. (SWAPs: 15, Depth: 223)

Target Circuit: adder_n28.qasm | Logical Qubits: 28 | Hardware Limits: 127
    [+] Seed 42 compiled successfully in 0.01s. (SWAPs: 65, Depth: 503)
    [+] Seed 43 compiled successfully in 0.02s. (SWAPs: 68, Depth: 445)
    [+] Seed 44 compiled successfully in 0.01s. (SWAPs: 59, Depth: 423)
    [+] Seed 45 compiled successfully in 0.01s. (SWAPs: 64, Depth: 430)
    [+] Seed 46 compiled successfully in 0.02s. (SWAPs: 68, Depth: 465)

Target Circuit: adder_n4.qasm | Logical Qubits: 4 | Hardware Limits: 127
    [+] Seed 42 compiled successfully in 0.01s. (SWAPs

## Merging CSV

In [4]:
import os
import sys
import numpy as np
import pandas as pd

def generate_comparison_matrix(csv_v1: str, csv_v2: str, output_csv: str):
    """
    Executes a rigorous statistical comparison between two transpilation benchmark datasets.
    Calculates absolute deltas and percentage regressions/improvements.
    """
    for file_path in [csv_v1, csv_v2]:
        if not os.path.isfile(file_path):
            sys.exit(f"FATAL: Required dataset '{file_path}' does not exist. Execution halted.")

    try:
        df1 = pd.read_csv(csv_v1)
        df2 = pd.read_csv(csv_v2)
    except Exception as e:
        sys.exit(f"FATAL: Data ingestion failed. Verify CSV integrity. Error: {e}")

    keys = ["Backend", "Circuit"]
    # UPDATED: "2Q Gates" replaced with "SWAP Count" to match routing-isolated outputs
    metrics = ["Execution Time (s)", "Total Gates", "SWAP Count", "Total Depth", "2Q Depth", "Successful_Runs"]
    
    for df, name in zip([df1, df2], [csv_v1, csv_v2]):
        missing = [col for col in keys + metrics if col not in df.columns]
        if missing:
            sys.exit(f"FATAL: Dataset '{name}' is missing required columns: {missing}")

    print(f"[*] Ingested {len(df1)} records from {csv_v1}")
    print(f"[*] Ingested {len(df2)} records from {csv_v2}")

    df_merged = pd.merge(df1, df2, on=keys, how="outer", suffixes=('_V1', '_V2'))

    for m in metrics:
        col_v1 = f"{m}_V1"
        col_v2 = f"{m}_V2"
        abs_diff_col = f"{m}_Abs_Delta"
        pct_diff_col = f"{m}_%_Change"

        df_merged[abs_diff_col] = df_merged[col_v2] - df_merged[col_v1]

        if m != "Successful_Runs":
            df_merged[pct_diff_col] = np.where(
                df_merged[col_v1] == 0,
                np.nan,
                (df_merged[abs_diff_col] / df_merged[col_v1]) * 100
            )

    final_cols = keys.copy()
    for m in metrics:
        final_cols.extend([f"{m}_V1", f"{m}_V2", f"{m}_Abs_Delta"])
        if m != "Successful_Runs":
            final_cols.append(f"{m}_%_Change")

    # FIXED: Added .copy() to enforce a new dataframe allocation, preventing SettingWithCopyWarning
    df_final = df_merged[final_cols].copy()

    float_cols = df_final.select_dtypes(include=['float64']).columns
    df_final[float_cols] = df_final[float_cols].round(2)

    try:
        df_final.to_csv(output_csv, index=False)
        print(f"[+] Dimensional comparison complete. Matrix written to: {output_csv}")
    except Exception as e:
        sys.exit(f"FATAL: Could not write output file '{output_csv}'. Error: {e}")

    # SUMMARY UPDATED: Now evaluating SWAP metrics specifically for routing tests
    regressions_depth = df_final[df_final["2Q Depth_%_Change"] > 0]
    improvements_depth = df_final[df_final["2Q Depth_%_Change"] < 0]
    
    regressions_swap = df_final[df_final["SWAP Count_%_Change"] > 0]
    improvements_swap = df_final[df_final["SWAP Count_%_Change"] < 0]
    
    print(f"\n--- Matrix Summary ---")
    print(f"Total Circuits Analyzed: {len(df_final)}")
    print(f"Circuits with 2Q Depth Regressions in V2 : {len(regressions_depth)}")
    print(f"Circuits with 2Q Depth Improvements in V2: {len(improvements_depth)}")
    print(f"Circuits with SWAP Count Regressions in V2 : {len(regressions_swap)}")
    print(f"Circuits with SWAP Count Improvements in V2: {len(improvements_swap)}")
    print(f"----------------------\n")

if __name__ == "__main__":
    # Ensure these file names match the exact outputs generated by your transpiler pipeline
    baseline_dataset = "averaged_routing_only_benchmark_2_4_1.csv"
    target_dataset = "averaged_routing_only_benchmark_2_5_0.csv"
    output_matrix = "qiskit_version_routing_only_comparison.csv"
    
    generate_comparison_matrix(
        csv_v1=baseline_dataset, 
        csv_v2=target_dataset, 
        output_csv=output_matrix
    )

[*] Ingested 176 records from averaged_routing_only_benchmark_2_4_1.csv
[*] Ingested 176 records from averaged_routing_only_benchmark_2_5_0.csv
[+] Dimensional comparison complete. Matrix written to: qiskit_version_routing_only_comparison.csv

--- Matrix Summary ---
Total Circuits Analyzed: 176
Circuits with 2Q Depth Regressions in V2 : 62
Circuits with 2Q Depth Improvements in V2: 28
Circuits with SWAP Count Regressions in V2 : 58
Circuits with SWAP Count Improvements in V2: 27
----------------------

